# `table_metric_extraction_and_pwc_benchmark_ollama.ipynb`

Ollama extraction (qwen3:1.7b, llama3:8b) and evaluation against **Metrics** in `pwc_final.json`.

Input: LightOnOCR table caches under `data/pdf_files_3`. Default corpus: `data/pdf_files_3`.

Outputs per model: `ollama_{model}_lightonocr_combinations_pdf_files.xlsx`.

Evaluation: `ollama_metric_eval_allpapers.xlsx`, `ollama_metric_audit_allpapers.xlsx`.

Flags: `FILTER_PREDICTIONS_TO_PWC_GT_VOCAB=False (minimal pipeline)`, `EVAL_ONLY_PWC_MATCHED_PAPERS=True`.

Replication: `table_extraction/README_pwc_benchmarks.md`.


## 1 — Configuration


In [ ]:
import ast
import json
import os
import re
import subprocess
import sys
import time
import hashlib
import difflib
import unicodedata
from pathlib import Path
from typing import Any

import pandas as pd
from bs4 import BeautifulSoup

AUTO_INSTALL = True
if AUTO_INSTALL:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "ollama>=0.4.8", "beautifulsoup4", "openpyxl"]
    )

def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, here.parent]:
        if (p / "data" / "pwc_final.json").is_file():
            return p
    raise FileNotFoundError("Repo root not found (expected data/pwc_final.json)")

REPO = _find_repo_root()
TABLE_EXTRACTION_DIR = REPO / "table_extraction"
CORPUS_TAG = "pdf_files"  # suffix in Excel filenames
PDF_FILES_DIR = REPO / "data" / "pdf_files_3"  # large corpus; use REPO / "data" / "pdf_files" for the small corpus
LLM_CACHE_DIR = TABLE_EXTRACTION_DIR / f"llm_ollama_cache_{CORPUS_TAG}"

# (ollama_model_tag, slug for filenames); match tags to `ollama list`
OLLAMA_MODELS: list[tuple[str, str]] = [
    ("qwen3:1.7b", "qwen3_1.7b"),
    ("llama3:8b", "llama3_8b"),
]

USE_LLM_RESPONSE_CACHE = True
MAX_CHARS_PER_CHUNK = 3500
DEBUG_LIMIT_CHUNKS: int | None = None  # e.g. 20 for a quick test; None = full corpus
MAX_PAPERS: int | None = None  # e.g. 5 for a smoke test

# Ollama HTTP: first chunk may load the model; increase if you see timeouts
OLLAMA_REQUEST_TIMEOUT = 1800.0
OLLAMA_DISABLE_THINKING = True  # required for qwen3; otherwise requests often time out
OLLAMA_MAX_RETRIES = 2
OLLAMA_RETRY_SLEEP_SEC = 10.0
OLLAMA_CHAT_OPTIONS = {"temperature": 0, "num_predict": 512}

PWC_GT_JSON = "pwc_final.json"
EVAL_OUTPUT_SUFFIX = "_minimal"
BERTSCORE_LANG = "en"
MATCH_THRESHOLD = 0.50
XML_FILES_DIR = Path("data/xml_files_3")
GT_XML_SECTION_KEYWORDS = []
GT_SCAN_FULL_BODY_IF_EMPTY = True
EVAL_SETUP_TABLES_ONLY = True
EVAL_SETUP_TEXT_ONLY = True
EVAL_SETUP_TABLES_PLUS_TEXT = True
# False = predicciones sin filtro al vocabulario Metrics de PwC por paper (precision mas baja)
FILTER_PREDICTIONS_TO_PWC_GT_VOCAB = False
# True = only papers with PwC GT (~165), same as GLiNER
EVAL_ONLY_PWC_MATCHED_PAPERS = True
# text_only: same tool on TEI narrative, not GROBID regex alone
TEXT_ONLY_INPUT = "tei_xml"  # narrativa GROBID en xml_files_3 (sin tablas HTML)

EVAL_OUTPUT_EXCEL = TABLE_EXTRACTION_DIR / f"evaluation_ollama_{CORPUS_TAG}.xlsx"
OLLAMA_EVAL_ALL_XLSX = TABLE_EXTRACTION_DIR / f"ollama_metric_eval_allpapers{EVAL_OUTPUT_SUFFIX}.xlsx"
OLLAMA_AUDIT_ALL_XLSX = TABLE_EXTRACTION_DIR / f"ollama_metric_audit_allpapers{EVAL_OUTPUT_SUFFIX}.xlsx"

# Wall-clock times for LLM inference (165 papers). If Section 5 used cache only
# (0 LLM calls), Section 6b uses these values for time_seconds in the eval Excel.
OLLAMA_REPORTED_INFERENCE_SECONDS: dict[tuple[str, str], float] = {
    ("qwen3:1.7b", "tables_only"): 3455.75,
    ("qwen3:1.7b", "text_only"): 162.27,
    ("qwen3:1.7b", "tables_plus_text"): 3618.02,
    ("llama3:8b", "tables_only"): 4873.41,
    ("llama3:8b", "text_only"): 258.40,
    ("llama3:8b", "tables_plus_text"): 5131.81,
}

MODE = "raw"  # reservado (blacklist como en GLiNER); LLM no usa blacklist por ahora
SHOW_VERBOSE = False

print(f"Repo: {REPO}")
print(f"PDF/cache dir: {PDF_FILES_DIR}")
print(f"Ollama models: {[m[0] for m in OLLAMA_MODELS]}")


## 2 — Check Ollama and models


In [ ]:
from ollama import Client

_host = os.environ.get("OLLAMA_HOST")
_client = Client(host=_host, timeout=30.0) if _host else Client(timeout=30.0)
_names: list[str] = []

try:
    _listed = _client.list()
    _names = [m.get("model", "") for m in (_listed.get("models") or [])]
    print("Ollama models on server:")
    for n in sorted(_names):
        print(f"  - {n}")
except Exception as e:
    print(f"Could not list Ollama models: {e}")
    print("Ensure Ollama is running: ollama serve")

for tag, _slug in OLLAMA_MODELS:
    if any(tag in n or n.startswith(tag.split(":")[0]) for n in _names):
        print(f"OK: {tag}")
    else:
        print(f"WARN: {tag} not in list — run: ollama pull {tag}")

## 3 — Load LightOnOCR caches (same input as GLiNER)


In [ ]:
ocr_outputs: dict[str, dict] = {}
cache_files = sorted(PDF_FILES_DIR.glob("*_lightonocr.json"))
if not cache_files:
    raise FileNotFoundError(f"No *_lightonocr.json in {PDF_FILES_DIR}")

stems = [c.name.replace("_lightonocr.json", "") for c in cache_files]
if MAX_PAPERS:
    stems = stems[:MAX_PAPERS]

for stem in stems:
    path = PDF_FILES_DIR / f"{stem}_lightonocr.json"
    with open(path, encoding="utf-8") as f:
        ocr_outputs[stem] = json.load(f)

n_tables = sum(
    len(t.get("tables", []))
    for res in ocr_outputs.values()
    for t in res.get("results", [])
)
print(f"Papers loaded: {len(ocr_outputs)}")
print(f"Tables total: {n_tables}")

## 4 — Ollama extractor and table helpers


In [ ]:
from ollama import Client

# ── Prompt + parser (equivalente a llm_extractors, pero dict model/dataset/metric) ──

KGE_ENTITY_PROMPT = (
    "You extract structured entities from knowledge graph embedding (KGE) paper tables.\n"
    "Given the table fragment below, return ONLY a Python dict with exactly these keys:\n"
    "  'model'    -> list of model / method names (e.g. TransE, RotatE, ComplEx)\n"
    "  'dataset'  -> list of benchmark datasets (e.g. FB15k-237, WN18RR, YAGO3-10)\n"
    "  'metric'   -> list of evaluation metrics (e.g. MRR, Hits@1, Hits@10, MR, AUC)\n"
    "Each value must be a list of strings. Use [] for keys with nothing found.\n"
    "Do not include numeric cell values, hyperparameters (learning rate), or column headers alone.\n"
    "Example: {'model': ['TransE'], 'dataset': ['FB15k-237'], 'metric': ['MRR', 'Hits@1']}\n"
)


def _parse_dict_response(raw: str) -> dict[str, list[str]]:
    text = (raw or "").strip()
    if not text:
        return {"model": [], "dataset": [], "metric": []}
    if "```" in text:
        m = re.search(r"```(?:python)?\s*([\s\S]*?)```", text, flags=re.IGNORECASE)
        if m:
            text = m.group(1).strip()
    parsed: Any = None
    try:
        parsed = ast.literal_eval(text)
    except (ValueError, SyntaxError):
        start, end = text.find("{"), text.rfind("}")
        if start >= 0 and end > start:
            try:
                parsed = ast.literal_eval(text[start : end + 1])
            except (ValueError, SyntaxError):
                parsed = None
    out: dict[str, list[str]] = {"model": [], "dataset": [], "metric": []}
    if not isinstance(parsed, dict):
        return out
    for key in ("model", "dataset", "metric"):
        val = parsed.get(key, [])
        if isinstance(val, str):
            val = [val] if val.strip() else []
        elif not isinstance(val, list):
            val = []
        out[key] = [str(x).strip() for x in val if str(x).strip()]
    return out


def _ollama_use_think_false(model_name: str) -> bool:
    base = model_name.split(":")[0].lower()
    return any(x in base for x in ("qwen3", "qwen2.5", "deepseek-r1"))


class OllamaKGEExtractor:
    """Ollama chat API — mismo rol que LlamaExtractor/QwenExtractor en llm_extractors.pyw."""

    def __init__(
        self,
        model_name: str,
        *,
        timeout: float | None = None,
        host: str | None = None,
        disable_thinking: bool | None = None,
    ):
        self.model_name = model_name
        self.host = host or os.environ.get("OLLAMA_HOST")
        self.timeout = float(timeout if timeout is not None else OLLAMA_REQUEST_TIMEOUT)
        self.disable_thinking = (
            OLLAMA_DISABLE_THINKING if disable_thinking is None else disable_thinking
        )
        kwargs: dict[str, Any] = {"timeout": self.timeout}
        if self.host:
            kwargs["host"] = self.host
        self.client = Client(**kwargs)
        think_note = ""
        if self.disable_thinking and _ollama_use_think_false(self.model_name):
            think_note = ", think=False"
        print(
            f"[Ollama] {self.model_name} ready "
            f"(host={self.host or 'default'}, timeout={self.timeout}s{think_note})"
        )

    def warmup(self) -> bool:
        """Carga el modelo en memoria antes del corpus (evita timeout en el 1er chunk)."""
        try:
            self.extract_entities("Reply with exactly: {'model': [], 'dataset': [], 'metric': []}")
            print(f"[Ollama] warmup OK: {self.model_name}")
            return True
        except Exception as e:
            print(f"[Ollama] warmup failed: {e}")
            return False

    def _build_messages(self, table_chunk: str) -> list[dict[str, str]]:
        return [
            {
                "role": "system",
                "content": (
                    "You are a strict information extraction assistant for ML benchmark tables. "
                    "Answer only with a valid Python dict as specified."
                ),
            },
            {"role": "user", "content": f"Table fragment:\n{table_chunk}"},
            {"role": "user", "content": KGE_ENTITY_PROMPT},
        ]

    def _chat_kwargs(self) -> dict[str, Any]:
        kw: dict[str, Any] = {"options": dict(OLLAMA_CHAT_OPTIONS)}
        if self.disable_thinking and _ollama_use_think_false(self.model_name):
            kw["think"] = False
        return kw

    def extract_entities(self, table_chunk: str) -> dict[str, list[str]] | None:
        """None = fallo de red/timeout (no cachear)."""
        if not str(table_chunk).strip():
            return {"model": [], "dataset": [], "metric": []}
        last_err: Exception | None = None
        for attempt in range(OLLAMA_MAX_RETRIES + 1):
            try:
                response = self.client.chat(
                    model=self.model_name,
                    messages=self._build_messages(table_chunk),
                    **self._chat_kwargs(),
                )
                content = (response.get("message") or {}).get("content", "")
                return _parse_dict_response(content)
            except Exception as e:
                last_err = e
                err_s = str(e).lower()
                retryable = "timed out" in err_s or "timeout" in err_s or "connection" in err_s
                if attempt < OLLAMA_MAX_RETRIES and retryable:
                    wait = OLLAMA_RETRY_SLEEP_SEC * (attempt + 1)
                    print(
                        f"[Ollama:{self.model_name}] {e} — retry {attempt + 1}/{OLLAMA_MAX_RETRIES} "
                        f"in {wait:.0f}s"
                    )
                    time.sleep(wait)
                    continue
                print(f"[Ollama:{self.model_name}] error: {e}")
                return None
        if last_err:
            print(f"[Ollama:{self.model_name}] error: {last_err}")
        return None

    def extract(self, text: str, question: str | None = None) -> list:
        """API compatible con llm_extractors: pregunta → lista Python."""
        if not question:
            return self.extract_entities(text).get("metric", [])
        messages = [
            {"role": "system", "content": "You are an assistant for QA tasks. Use only provided context."},
            {"role": "user", "content": f"Context chunk: {text}"},
            {
                "role": "user",
                "content": (
                    f"Given the following question: {question}\n"
                    "Return the answer only in a Python list format, i.e. ['A','B']. "
                    "You must return an empty list if there is no answer."
                ),
            },
        ]
        try:
            response = self.client.chat(
                model=self.model_name, messages=messages, **self._chat_kwargs()
            )
            raw = (response.get("message") or {}).get("content", "").strip()
            parsed = ast.literal_eval(raw)
            if isinstance(parsed, list):
                return [] if parsed == [""] else parsed
        except Exception as e:
            print(f"[Ollama:{self.model_name}] extract() error: {e}")
        return []


# ── Helpers tabla (mismo flujo que GLiNER) ──

LABELS = ("model", "dataset", "metric")


def _header_and_rows_from_html(html: str) -> tuple[list[str], list[str]]:
    soup = BeautifulSoup(html, "html.parser")
    thead = soup.find("thead")
    tbody = soup.find("tbody")

    def _row_text(tr):
        cells = [c.get_text(separator=" ", strip=True) for c in tr.find_all(["th", "td"])]
        cells = [c for c in cells if c]
        return " | ".join(cells) if cells else None

    header_lines, body_lines = [], []
    if thead is not None:
        for tr in thead.find_all("tr"):
            txt = _row_text(tr)
            if txt:
                header_lines.append(txt)
    trs = tbody.find_all("tr") if tbody is not None else soup.find_all("tr")
    for tr in trs:
        if thead is not None and tr in thead.find_all("tr"):
            continue
        cells = tr.find_all(["th", "td"])
        if not cells:
            continue
        txt = _row_text(tr)
        if not txt:
            continue
        if all(c.name == "th" for c in cells) and not body_lines:
            header_lines.append(txt)
        else:
            body_lines.append(txt)
    if not header_lines and body_lines:
        header_lines, body_lines = [body_lines[0]], body_lines[1:]
    return header_lines, body_lines


# Caracteres de control que openpyxl rechaza (p. ej. BEL \x07 en LaTeX mal parseado)
_EXCEL_ILLEGAL_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")


def _sanitize_excel_text(value: Any) -> str:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    s = _EXCEL_ILLEGAL_RE.sub("", str(value))
    return s[:32767]


def _sanitize_df_for_excel(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        if out[col].dtype == object or pd.api.types.is_string_dtype(out[col]):
            out[col] = out[col].map(_sanitize_excel_text)
    return out


def _clean_entity(value: str) -> str:
    s = _sanitize_excel_text(str(value).strip())
    s = re.sub(r"\[[^\]]{1,50}\]", "", s)
    s = re.sub(r"\$([^$]+?)\$", r"\1", s)
    s = re.sub(r"\s+", " ", s).strip(" .,:;-")
    if re.fullmatch(r"[+-]?\d+(?:\.\d+)?", s):
        return ""
    return s if len(s) >= 2 else ""


def _merge_entity_sets(target: dict[str, set[str]], other: dict[str, list[str]]) -> None:
    for label in LABELS:
        for raw in other.get(label, []) or []:
            v = _clean_entity(raw)
            if v:
                target[label].add(v)


def _combinations_from_sets(ents: dict[str, set[str]]) -> list[tuple[str, str, str]]:
    if not ents["metric"]:
        return []
    models = sorted(ents["model"]) or [""]
    datasets = sorted(ents["dataset"]) or [""]
    metrics = sorted(ents["metric"])
    out = []
    for m in models:
        for d in datasets:
            for mt in metrics:
                if m or d:
                    out.append((m, d, mt))
    return out


def _chunk_cache_path(model_slug: str, chunk_id: str) -> Path:
    d = LLM_CACHE_DIR / model_slug
    d.mkdir(parents=True, exist_ok=True)
    return d / f"{chunk_id}.json"


def _chunk_id(table_name: str, kind: str, idx: int, text: str) -> str:
    h = hashlib.sha256(text.encode("utf-8", errors="replace")).hexdigest()[:16]
    return f"{table_name}_{kind}_{idx}_{h}"


print("Table + LLM helpers ready.")

## 5 — Ollama extraction per model

Run the PwC helpers code cell (Section 5a), then the cell that defines `run_ollama_*` (Section 5b), then the extraction loop (Section 5c).


Run the next code cell before the extraction loop. It defines PwC matching, BERTScore evaluation, and TEI helpers (`pwc_final.json` GT only).


In [ ]:
import torch
from bert_score import score as bert_score_fn
_BERTSCORE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
_BERTSCORE_LANG = globals().get("BERTSCORE_LANG", "en")

MATCH_THRESHOLD = 0.50

def _strip_accents(text: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFKD", text) if not unicodedata.combining(ch))


def normalize_text(text: object) -> str:
    if text is None:
        return ""
    s = str(text).strip().lower()
    s = _strip_accents(s)
    s = re.sub(r"\s+", " ", s)
    return s


def normalize_paper(text: object) -> str:
    s = normalize_text(text)
    if s.startswith("http"):
        s = s.rstrip("/")
        s = s.split("/")[-1]
    s = re.sub(r"[^a-z0-9]+", " ", s).strip()
    return s


def normalize_dataset(text: object) -> str:
    s = normalize_text(text)
    s = s.replace(" ", "").replace("_", "").replace("-", "")
    return s


def normalize_metric(text: object) -> str:
    s = normalize_text(text)
    if not s:
        return ""
    s = s.replace("%", "").replace("(", "").replace(")", "")
    s = re.sub(r"[^a-z0-9@]+", "", s)
    if s.endswith("s") and len(s) > 3 and s[-2].isalpha():
        s = s[:-1]
    return s


def safe_f1(p: float, r: float) -> float:
    return 0.0 if (p + r) == 0 else (2 * p * r) / (p + r)


def _pwc_metrics_list(obj: dict) -> list:
    """Metric names from pwc_final (`Metrics` or legacy `metrics`)."""
    raw = obj.get("metrics")
    if raw is None or (isinstance(raw, list) and len(raw) == 0):
        raw = obj.get("Metrics")
    if not isinstance(raw, list):
        return []
    return [str(m).strip() for m in raw if str(m).strip()]


def _arxiv_norm_from_text(text: object) -> str:
    if text is None:
        return ""
    s = str(text).strip().lower()
    m = re.search(r"(\d{4}\.\d{4,5})(?:v\d+)?", s)
    return m.group(1) if m else ""


def _paperswithcode_slug_from_url(url: object) -> str:
    if url is None:
        return ""
    s = str(url).strip().rstrip("/")
    if "paperswithcode.com/paper/" in s:
        return s.split("paperswithcode.com/paper/", 1)[-1].split("/")[0].strip().lower()
    return ""


def _pdf_stem_slug_from_title(title: object, max_len: int = 50) -> str:
    """Slug like pdf_files_* stems: normalized title → words joined with '_' (truncated)."""
    pn = normalize_paper(title)
    if not pn:
        return ""
    return re.sub(r"\s+", "_", pn)[:max_len]


def _stem_matches_title_slug(stem: str, title_slug: str) -> bool:
    if not stem or not title_slug:
        return False
    a, b = stem.lower(), title_slug.lower()
    return a == b or b.startswith(a) or a.startswith(b)


def _pwc_identity_from_row(mrow: dict | None, paper_raw: str) -> tuple[str, str]:
    """Stable paper key aligned with dataset JSON: arxiv > PapersWithCode slug > normalized title."""
    mr = mrow if isinstance(mrow, dict) else {}
    for k in ("arxiv_id", "arxiv", "paper_arxiv_id"):
        a = _arxiv_norm_from_text(mr.get(k))
        if a:
            return f"arxiv:{a}", "arxiv"
    for ukey in ("paper_url", "url"):
        sl = _paperswithcode_slug_from_url(mr.get(ukey))
        if sl:
            return f"slug:{sl}", "slug"
    blob = " ".join(str(mr.get(x, "") or "") for x in ("paper_title", "paper", "paper_url"))
    mb = re.search(r"\b(\d{4}\.\d{5})\b", blob) or re.search(r"\b(\d{4}\.\d{4})\b", blob)
    if mb:
        return f"arxiv:{mb.group(1)}", "arxiv"
    a2 = _arxiv_norm_from_text(paper_raw)
    if a2:
        return f"arxiv:{a2}", "arxiv"
    sl2 = _paperswithcode_slug_from_url(paper_raw)
    if sl2:
        return f"slug:{sl2}", "slug"
    return f"title:{normalize_paper(paper_raw)}", "title"


def _dataset_json_identity(o: dict) -> tuple[str, str]:
    for k in ("arxiv_id", "url_abs", "url_pdf", "paper_url"):
        a = _arxiv_norm_from_text(o.get(k))
        if a:
            return f"arxiv:{a}", "arxiv"
    sl = _paperswithcode_slug_from_url(o.get("paper_url"))
    if sl:
        return f"slug:{sl}", "slug"
    return f"title:{normalize_paper(o.get('title'))}", "title"


def _pwc_final_lookup_maps(papers: list) -> tuple[dict, dict, dict, dict, dict]:
    by_arxiv: dict[str, dict] = {}
    by_slug: dict[str, dict] = {}
    by_title: dict[str, dict] = {}
    by_pdf_stem: dict[str, dict] = {}
    by_title_stem: dict[str, dict] = {}
    for o in papers:
        if not isinstance(o, dict):
            continue
        for k in ("arxiv_id", "url_abs", "url_pdf"):
            a = _arxiv_norm_from_text(o.get(k))
            if a:
                by_arxiv.setdefault(a, o)
        sl = _paperswithcode_slug_from_url(o.get("paper_url"))
        if sl:
            by_slug.setdefault(sl, o)
        pn = normalize_paper(o.get("title"))
        if pn:
            by_title.setdefault(pn, o)
        ts = _pdf_stem_slug_from_title(o.get("title"))
        if ts:
            by_title_stem.setdefault(ts, o)
        lp = str(o.get("local_pdf_path") or "").replace("\\", "/")
        if lp:
            by_pdf_stem.setdefault(Path(lp).stem, o)
    return by_arxiv, by_slug, by_title, by_pdf_stem, by_title_stem


def _match_pwc_entry_for_pdf_stem(
    stem: str,
    by_arxiv: dict,
    by_title: dict,
    by_pdf_stem: dict,
    threshold: float,
    *,
    by_title_stem: dict | None = None,
) -> tuple[dict | None, str]:
    if stem in by_pdf_stem:
        return by_pdf_stem[stem], "local_pdf_path"
    a = _arxiv_norm_from_text(stem)
    if a and a in by_arxiv:
        return by_arxiv[a], "arxiv_id"
    by_title_stem = by_title_stem or {}
    stem_l = stem.lower()
    stem_slug = _pdf_stem_slug_from_title(stem.replace("_", " "))
    if stem in by_title_stem:
        return by_title_stem[stem], "title_stem_exact"
    for slug, obj in by_title_stem.items():
        if slug.lower() == stem_l or (stem_slug and slug.lower() == stem_slug.lower()):
            return obj, "title_stem_exact"
    prefix_hits: list[tuple[int, dict, str]] = []
    for slug, obj in by_title_stem.items():
        if _stem_matches_title_slug(stem_slug or stem, slug):
            prefix_hits.append((len(slug), obj, slug))
    if prefix_hits:
        prefix_hits.sort(key=lambda x: x[0], reverse=True)
        best_len, best_o, best_slug = prefix_hits[0]
        ref = (stem_slug or stem).lower()
        if best_slug.lower().startswith(ref) or len(prefix_hits) == 1:
            return best_o, "title_stem_prefix"
    pn = normalize_paper(stem.replace("_", " "))
    if pn in by_title:
        return by_title[pn], "title_exact"
    best_o, best_sc = None, 0.0
    for t, o in by_title.items():
        sc = difflib.SequenceMatcher(None, pn, t).ratio()
        if sc > best_sc:
            best_sc, best_o = sc, o
    if best_o and best_sc >= threshold:
        return best_o, "title_fuzzy"
    return None, "unmatched"


def build_ground_truth_for_pdf_corpus(
    repo_root: Path,
    json_basename: str,
    pdf_rel_dir: Path,
    *,
    match_threshold: float,
    skip_empty_metrics: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """GT metric names from JSON `metrics`, one row per PDF in the corpus folder."""
    jp = (repo_root / "data" / json_basename).resolve()
    if not jp.is_file():
        raise FileNotFoundError(jp)
    with open(jp, encoding="utf-8") as f:
        papers = json.load(f)
    if not isinstance(papers, list):
        raise ValueError(f"Expected a JSON list in {jp}")

    pdf_dir = (repo_root / pdf_rel_dir).resolve()
    if not pdf_dir.is_dir():
        raise FileNotFoundError(pdf_dir)

    by_arxiv, _by_slug, by_title, by_pdf_stem, by_title_stem = _pwc_final_lookup_maps(papers)
    metric_rows: list[dict] = []
    audit_rows: list[dict] = []
    corpus_rows: list[dict] = []

    for pdf_path in sorted(pdf_dir.glob("*.pdf")):
        stem = pdf_path.stem
        obj, how = _match_pwc_entry_for_pdf_stem(
            stem, by_arxiv, by_title, by_pdf_stem, match_threshold,
            by_title_stem=by_title_stem,
        )
        if obj is None:
            corpus_rows.append(
                {
                    "pdf_stem": stem,
                    "pwc_match": "unmatched",
                    "match_method": "",
                    "title": "",
                    "arxiv_id": "",
                    "n_gt_metrics": 0,
                }
            )
            continue
        title = str(obj.get("title") or "").strip()
        pn = normalize_paper(title)
        mets: set[str] = set()
        for m in _pwc_metrics_list(obj):
            nm = normalize_metric(str(m))
            if nm:
                mets.add(nm)
        corpus_rows.append(
            {
                "pdf_stem": stem,
                "pwc_match": "matched",
                "match_method": how,
                "title": title,
                "arxiv_id": str(obj.get("arxiv_id") or ""),
                "n_gt_metrics": len(mets),
            }
        )
        audit_rows.append(
            {
                "title": title,
                "paper_norm": pn,
                "arxiv_id": str(obj.get("arxiv_id") or ""),
                "pdf_stem": stem,
                "n_gt_metrics": len(mets),
                "gt_metrics_flat": ", ".join(sorted(mets)),
            }
        )
        if skip_empty_metrics and not mets:
            continue
        if not pn:
            continue
        mk, mk_kind = _dataset_json_identity(obj)
        for m in sorted(mets):
            metric_rows.append(
                {
                    "paper_raw": title,
                    "dataset_raw": "",
                    "metric_raw": m,
                    "paper_norm": pn,
                    "metric_norm": m,
                    "paper_match_key": mk,
                    "paper_match_kind": mk_kind,
                }
            )

    cols = [
        "paper_raw",
        "dataset_raw",
        "metric_raw",
        "paper_norm",
        "metric_norm",
        "paper_match_key",
        "paper_match_kind",
    ]
    gt_df = pd.DataFrame(metric_rows).drop_duplicates() if metric_rows else pd.DataFrame(columns=cols)
    return gt_df, pd.DataFrame(audit_rows), pd.DataFrame(corpus_rows)


def load_ours_combinations(path: Path, sheet_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Input Excel not found: {path}")
    df = pd.read_excel(path, sheet_name=sheet_name)
    colmap = {c.lower(): c for c in df.columns}
    missing = {"paper", "dataset", "metric"} - set(colmap.keys())
    if missing:
        raise ValueError(f"Sheet '{sheet_name}' missing columns: {missing}")
    out = pd.DataFrame({
        "paper_raw": df[colmap["paper"]],
        "dataset_raw": df[colmap["dataset"]],
        "metric_raw": df[colmap["metric"]],
    })
    out["paper_norm"] = out["paper_raw"].map(normalize_paper)
    out["dataset_norm"] = out["dataset_raw"].map(normalize_dataset)
    out["metric_norm"] = out["metric_raw"].map(normalize_metric)
    out = out[(out["paper_norm"] != "") & (out["metric_norm"] != "")]
    return out.drop_duplicates()


def load_pred_tables_only_sheet(excel_path: Path) -> pd.DataFrame:
    """Table pipeline only: Excel sheet Tables With Metrics (no Combinations)."""
    cols = [
        "paper_raw", "dataset_raw", "metric_raw",
        "paper_norm", "dataset_norm", "metric_norm",
    ]
    try:
        df = pd.read_excel(excel_path, sheet_name="Tables With Metrics")
    except ValueError:
        return pd.DataFrame(columns=cols)
    colmap = {c.lower(): c for c in df.columns}
    if "paper" not in colmap or "metric" not in colmap:
        return pd.DataFrame(columns=cols)
    part = pd.DataFrame(
        {
            "paper_raw": df[colmap["paper"]],
            "dataset_raw": df[colmap["dataset"]] if "dataset" in colmap else "",
            "metric_raw": df[colmap["metric"]],
        }
    )
    part["paper_norm"] = part["paper_raw"].map(normalize_paper)
    part["dataset_norm"] = part["dataset_raw"].map(normalize_dataset)
    part["metric_norm"] = part["metric_raw"].map(normalize_metric)
    part = part[(part["paper_norm"] != "") & (part["metric_norm"] != "")]
    return part.drop_duplicates()


def enrich_ours_combinations_with_dataset_json(
    ours_df: pd.DataFrame,
    json_path: Path,
    *,
    match_threshold: float | None = None,
) -> pd.DataFrame:
    """Add `paper_match_key` aligned with GT: PDF stem / arxiv id → `_dataset_json_identity`."""
    if not json_path.is_file():
        raise FileNotFoundError(json_path)
    thr = MATCH_THRESHOLD if match_threshold is None else match_threshold
    with open(json_path, encoding="utf-8") as f:
        papers = json.load(f)
    if not isinstance(papers, list):
        raise ValueError(f"Expected a JSON list in {json_path}")

    by_arxiv, _by_slug, by_title, by_pdf_stem, by_title_stem = _pwc_final_lookup_maps(papers)
    out = ours_df.copy()
    keys: list[str] = []
    kinds: list[str] = []
    for _, row in out.iterrows():
        raw = str(row.get("paper_raw", "") or "").strip()
        obj, _how = _match_pwc_entry_for_pdf_stem(
            raw, by_arxiv, by_title, by_pdf_stem, thr, by_title_stem=by_title_stem
        )
        if obj is not None:
            mk, kind = _dataset_json_identity(obj)
        else:
            mk, kind = _pwc_identity_from_row(None, raw)
        keys.append(mk)
        kinds.append(kind)
    out["paper_match_key"] = keys
    out["paper_match_kind"] = kinds
    return out


def evaluate_combinations_id_aware(
    ours_df: pd.DataFrame,
    pwc_df: pd.DataFrame,
    threshold_title_fallback: float,
) -> dict[str, pd.DataFrame]:
    """Compare metric names grouped by `paper_match_key` (arxiv:/slug:/title:) instead of title-only fuzzy match."""
    if "paper_match_key" not in ours_df.columns or "paper_match_key" not in pwc_df.columns:
        raise ValueError("Expected column paper_match_key in both DataFrames.")

    def _agg_metric_set(s: pd.Series) -> set[str]:
        out: set[str] = set()
        for x in s.dropna().tolist():
            if str(x).strip():
                out.add(str(x))
        return out

    ours_g = ours_df.groupby("paper_match_key", sort=False)["metric_norm"].apply(_agg_metric_set)
    pwc_g = pwc_df.groupby("paper_match_key", sort=False)["metric_norm"].apply(_agg_metric_set)

    pwc_key_list = list(pwc_g.index)
    used_pwc: set[str] = set()
    key_map: dict[str, str] = {}
    for ok in ours_g.index:
        if ok in pwc_g.index:
            key_map[ok] = ok
            used_pwc.add(ok)
            continue
        if not str(ok).startswith("title:"):
            key_map[ok] = ""
            continue
        best_pk, best_sc = "", 0.0
        for pk in pwc_key_list:
            if pk in used_pwc:
                continue
            if not str(pk).startswith("title:"):
                continue
            sc = difflib.SequenceMatcher(None, ok, pk).ratio()
            if sc > best_sc:
                best_sc, best_pk = sc, pk
        if best_pk and best_sc >= threshold_title_fallback:
            key_map[ok] = best_pk
            used_pwc.add(best_pk)
        else:
            key_map[ok] = ""

    rep_norm = ours_df.groupby("paper_match_key", sort=False)["paper_norm"].first()

    per_paper_rows = []
    tp_total = fp_total = fn_total = 0
    for ok in ours_g.index:
        ours_m = set(ours_g[ok])
        pk = key_map.get(ok, "")
        pwc_m = set(pwc_g[pk]) if pk and pk in pwc_g.index else set()
        tp = len(ours_m & pwc_m)
        fp = len(ours_m - pwc_m)
        fn = len(pwc_m - ours_m)
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = safe_f1(precision, recall)
        tp_total += tp
        fp_total += fp
        fn_total += fn
        if pk and pk in pwc_g.index:
            how = "exact_key" if ok == pk else "fuzzy_title_key"
        else:
            how = "unmatched"
        per_paper_rows.append(
            {
                "paper_norm": rep_norm.get(ok, ""),
                "ours_match_key": ok,
                "matched_pwc_match_key": pk,
                "match_kind": how,
                "num_pred_metrics": len(ours_m),
                "num_gt_metrics": len(pwc_m),
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "precision": round(precision, 4),
                "recall": round(recall, 4),
                "f1": round(f1, 4),
                "pred_only_metrics": ", ".join(sorted(ours_m - pwc_m)),
                "gt_only_metrics": ", ".join(sorted(pwc_m - ours_m)),
            }
        )

    per_paper_df = pd.DataFrame(per_paper_rows).sort_values(
        ["f1", "recall", "precision"], ascending=[True, True, True]
    )

    micro_p = tp_total / (tp_total + fp_total) if (tp_total + fp_total) else 0.0
    micro_r = tp_total / (tp_total + fn_total) if (tp_total + fn_total) else 0.0
    micro_f1 = safe_f1(micro_p, micro_r)
    macro_p = float(per_paper_df["precision"].mean()) if len(per_paper_df) else 0.0
    macro_r = float(per_paper_df["recall"].mean()) if len(per_paper_df) else 0.0
    macro_f1 = float(per_paper_df["f1"].mean()) if len(per_paper_df) else 0.0

    matched_count = int((per_paper_df["match_kind"] != "unmatched").sum())
    ours_keys = set(ours_g.index)
    pwc_keys = set(pwc_g.index)
    mapped_pwc = {key_map[k] for k in ours_keys if key_map.get(k)}
    unmatched_ours_keys = sorted(k for k in ours_keys if not key_map.get(k))

    paper_matches_df = pd.DataFrame(
        {
            "ours_paper_norm": per_paper_df["paper_norm"],
            "ours_match_key": per_paper_df["ours_match_key"],
            "pwc_match_key": per_paper_df["matched_pwc_match_key"],
            "match_method": per_paper_df["match_kind"],
            "match_score": per_paper_df["match_kind"].map(lambda x: 1.0 if x == "exact_key" else (0.9 if x == "fuzzy_title_key" else 0.0)),
        }
    )

    global_df = pd.DataFrame(
        [
            {"metric": "eval_mode", "value": "arxiv_slug_title_keys"},
            {"metric": "micro_precision", "value": round(micro_p, 4)},
            {"metric": "micro_recall", "value": round(micro_r, 4)},
            {"metric": "micro_f1", "value": round(micro_f1, 4)},
            {"metric": "macro_precision", "value": round(macro_p, 4)},
            {"metric": "macro_recall", "value": round(macro_r, 4)},
            {"metric": "macro_f1", "value": round(macro_f1, 4)},
            {"metric": "papers_ours_total", "value": len(ours_keys)},
            {"metric": "papers_pwc_total", "value": len(pwc_keys)},
            {"metric": "papers_matched", "value": matched_count},
            {"metric": "papers_unmatched_ours", "value": int(per_paper_df["match_kind"].eq("unmatched").sum())},
            {"metric": "papers_unmatched_pwc", "value": len(pwc_keys - mapped_pwc)},
            {"metric": "tp_total", "value": tp_total},
            {"metric": "fp_total", "value": fp_total},
            {"metric": "fn_total", "value": fn_total},
        ]
    )

    err_rows = []
    for _, row in per_paper_df.iterrows():
        for m in filter(None, [x.strip() for x in str(row["pred_only_metrics"]).split(",")]):
            err_rows.append({"paper_norm": row["paper_norm"], "error_type": "FP_metric", "metric": m})
        for m in filter(None, [x.strip() for x in str(row["gt_only_metrics"]).split(",")]):
            err_rows.append({"paper_norm": row["paper_norm"], "error_type": "FN_metric", "metric": m})

    return {
        "paper_matches": paper_matches_df,
        "per_paper_scores": per_paper_df,
        "global_scores": global_df,
        "errors": pd.DataFrame(err_rows),
        "unmatched_ours": pd.DataFrame({"paper_match_key": unmatched_ours_keys}),
        "unmatched_pwc": pd.DataFrame({"paper_match_key": sorted(pwc_keys - mapped_pwc)}),
    }




def _resolve_xml_for_pdf_stem(
    repo_root: Path,
    pdf_stem: str,
    *,
    xml_rel_dir: Path | None = None,
) -> Path | None:
    """Resolve GROBID XML for a PDF stem (xml_files_3: <stem>.tei.xml)."""
    stem = str(pdf_stem).strip()
    if not stem:
        return None
    xml_dir = (repo_root / (xml_rel_dir or Path("data/xml_files_3"))).resolve()
    if not xml_dir.is_dir():
        return None
    direct = xml_dir / f"{stem}.tei.xml"
    if direct.is_file():
        return direct
    a = _arxiv_norm_from_text(stem)
    if a:
        hits = sorted(xml_dir.glob(f"*{a}*.xml"))
        if hits:
            return hits[0].resolve()
    return None
_PRED_DF_COLS = [
    "paper_raw", "dataset_raw", "metric_raw",
    "paper_norm", "dataset_norm", "metric_norm",
]



def merge_tables_and_text_predictions(
    tables_df: pd.DataFrame, text_df: pd.DataFrame
) -> pd.DataFrame:
    """Union of table and XML-text predictions (all corpus papers)."""
    parts = [df for df in (tables_df, text_df) if df is not None and not df.empty]
    if not parts:
        return pd.DataFrame(columns=_PRED_DF_COLS)
    return pd.concat(parts, ignore_index=True).drop_duplicates()


def _lookup_xml_text_metrics_for_paper(
    paper_raw: str,
    paper_norm: str,
    xml_metrics: dict[str, set[str]],
    threshold: float,
) -> tuple[set[str], str]:
    stem = str(paper_raw).strip()
    for key in (stem, paper_norm, normalize_paper(stem.replace("_", " "))):
        if key and key in xml_metrics:
            return set(xml_metrics[key]), "xml_sections"
    if xml_metrics:
        mk = _match_paper_norm_to_dict_key(paper_norm, list(xml_metrics.keys()), threshold)
        if mk:
            return set(xml_metrics.get(mk, set())), "xml_sections"
    return set(), ""


_SETUP_SOURCE_LABELS_BASE = {
    "tables_only": "LightOnOCR tables + Ollama (Tables With Metrics)",
    "text_only": "GROBID TEI narrative (xml_files_3)",
    "tables_plus_text": "tables_only ∪ text_only",
}


def _setup_source_label(mode: str, ollama_model: str | None = None) -> str:
    if not ollama_model:
        return _SETUP_SOURCE_LABELS_BASE.get(mode, mode)
    if mode == "tables_only":
        return f"LightOnOCR tables + Ollama ({ollama_model})"
    if mode == "text_only":
        return f"GROBID TEI narrative + Ollama ({ollama_model})"
    if mode == "tables_plus_text":
        return f"tables_only ∪ text_only [Ollama: {ollama_model}]"
    return _SETUP_SOURCE_LABELS_BASE.get(mode, mode)


def _global_score_value(global_df: pd.DataFrame, metric_name: str):
    s = global_df.loc[global_df["metric"] == metric_name, "value"]
    if s.empty:
        return None
    v = s.iloc[0]
    try:
        return round(float(v), 4)
    except (TypeError, ValueError):
        return v


def evaluate_pred_df_vs_pwc(
    repo_root: Path,
    pred_df: pd.DataFrame,
    pdf_rel_dir: Path | str,
    *,
    mode: str,
    prediction_source: str,
    json_basename: str = "pwc_final.json",
    match_threshold: float = MATCH_THRESHOLD,
    gt_df: pd.DataFrame | None = None,
) -> tuple[dict[str, pd.DataFrame], dict]:
    repo_root = repo_root.resolve()
    pdf_rel = Path(pdf_rel_dir)
    if gt_df is None:
        gt_df, _, _ = build_ground_truth_for_pdf_corpus(
            repo_root, json_basename, pdf_rel,
            match_threshold=match_threshold, skip_empty_metrics=True,
        )
    gt_json = (repo_root / "data" / json_basename).resolve()
    pred_e = enrich_ours_combinations_with_dataset_json(
        pred_df, gt_json, match_threshold=match_threshold
    )
    if not gt_df.empty and "paper_match_key" in gt_df.columns:
        gt_keys = set(gt_df["paper_match_key"].dropna().astype(str))
        pred_scored = pred_e[pred_e["paper_match_key"].isin(gt_keys)]
    else:
        pred_scored = pred_e
    results = evaluate_combinations_id_aware(
        pred_e, gt_df, threshold_title_fallback=match_threshold
    )
    g = results["global_scores"]
    return results, {
        "gt_json": json_basename,
        "corpus": str(pdf_rel),
        "mode": mode,
        "prediction_source": prediction_source,
        "paper_match_mode": _global_score_value(g, "eval_mode") or "arxiv_slug_title_keys",
        "micro_precision": _global_score_value(g, "micro_precision") or 0.0,
        "micro_recall": _global_score_value(g, "micro_recall") or 0.0,
        "micro_f1": _global_score_value(g, "micro_f1") or 0.0,
        "macro_precision": _global_score_value(g, "macro_precision"),
        "macro_recall": _global_score_value(g, "macro_recall"),
        "macro_f1": _global_score_value(g, "macro_f1"),
        "papers_pred_total": _global_score_value(g, "papers_ours_total"),
        "papers_gt_total": _global_score_value(g, "papers_pwc_total"),
        "papers_matched": _global_score_value(g, "papers_matched"),
        "tp_total": _global_score_value(g, "tp_total"),
        "fp_total": _global_score_value(g, "fp_total"),
        "fn_total": _global_score_value(g, "fn_total"),
        "pred_rows": len(pred_scored),
        "pred_rows_loaded": len(pred_df),
        "papers_pred_scored": int(pred_scored["paper_match_key"].nunique()) if len(pred_scored) else 0,
        "gliner_seconds": None,
        "xml_text_seconds": 0.0,
        "ocr_seconds": 0.0,
    }


def apply_ollama_run_timing(
    summary: dict,
    *,
    ollama_model: str | None,
    ollama_seconds: float | None,
    ollama_chunk_calls: int | None,
    xml_text_seconds: float | None = None,
) -> dict:
    summary = dict(summary)
    if ollama_model:
        summary["ollama_model"] = ollama_model
    if ollama_seconds is not None:
        summary["ollama_seconds"] = round(float(ollama_seconds), 3)
        summary["gliner_seconds"] = round(float(ollama_seconds), 3)
    if ollama_chunk_calls is not None:
        summary["ollama_chunk_calls"] = int(ollama_chunk_calls)
    if xml_text_seconds is not None:
        summary["xml_text_seconds"] = round(float(xml_text_seconds), 3)
    summary.setdefault("ocr_seconds", 0.0)
    return summary


def write_ollama_evaluation_excel(
    output_path: Path,
    *,
    mode_summary_rows: list[dict],
    pipeline_timing_rows: list[dict],
    papers_without_metrics: pd.DataFrame | None = None,
    text_only_audit_df: pd.DataFrame | None = None,
) -> Path:
    output_path = output_path.resolve()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    pwm = papers_without_metrics if papers_without_metrics is not None else pd.DataFrame()
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        pwm.to_excel(writer, sheet_name="papers_without_metrics", index=False)
        pd.DataFrame(pipeline_timing_rows).to_excel(writer, sheet_name="pipeline_timing", index=False)
        _ms = pd.DataFrame(mode_summary_rows)
        if not _ms.empty:
            _ms["precision"] = _ms.get("micro_precision", 0.0)
            _ms["recall"] = _ms.get("micro_recall", 0.0)
            _ms["f1"] = _ms.get("micro_f1", 0.0)
            _drop = [
                "micro_precision", "micro_recall", "micro_f1",
                "macro_precision", "macro_recall", "macro_f1",
            ]
            _ms = _ms.drop(columns=[c for c in _drop if c in _ms.columns])
        _ms.to_excel(writer, sheet_name="dataset_gt_mode_summary", index=False)
        if text_only_audit_df is not None and not text_only_audit_df.empty:
            text_only_audit_df.to_excel(writer, sheet_name="text_only_audit", index=False)
    return output_path


# --- Alineado con table_metric_extraction_and_pwc_benchmark_gliner.ipynb ---


def _df_to_pred_metric(df_like: pd.DataFrame) -> pd.DataFrame:
    cols = ["paper_raw", "dataset_raw", "metric_raw", "paper_norm", "dataset_norm", "metric_norm"]
    if df_like is None or df_like.empty:
        return pd.DataFrame(columns=cols)
    rows = []
    for _, r in df_like.iterrows():
        paper = str(r.get("paper", "") or r.get("paper_raw", "")).strip()
        metric_raw = str(r.get("metric", "") or r.get("metric_raw", "")).strip()
        nm = normalize_metric(metric_raw)
        if not paper or not nm:
            continue
        rows.append({
            "paper_raw": paper,
            "dataset_raw": str(r.get("dataset", "") or r.get("dataset_raw", "")),
            "metric_raw": metric_raw,
            "paper_norm": normalize_paper(paper),
            "dataset_norm": normalize_dataset(str(r.get("dataset", "") or r.get("dataset_raw", ""))),
            "metric_norm": nm,
        })
    out = pd.DataFrame(rows)
    if out.empty:
        return pd.DataFrame(columns=cols)
    out = out[(out["paper_norm"] != "") & (out["metric_norm"] != "")]
    return out.drop_duplicates()


def _agg_metric_list(s: pd.Series) -> list[str]:
    seen: set[str] = set()
    out: list[str] = []
    for x in s.dropna().tolist():
        sx = str(x).strip()
        if sx and sx not in seen:
            seen.add(sx)
            out.append(sx)
    return out


def _agg_dataset_list(s: pd.Series) -> list[str]:
    return _agg_metric_list(s)


def _filter_pred_df_to_gt_corpus(
    pred_df: pd.DataFrame,
    gt_df: pd.DataFrame,
    gt_json_path: Path,
    match_threshold: float,
) -> pd.DataFrame:
    """Keep predictions only for papers that match PwC GT (EVAL_ONLY_PWC_MATCHED_PAPERS)."""
    cols = ["paper_raw", "dataset_raw", "metric_raw", "paper_norm", "dataset_norm", "metric_norm"]
    if pred_df is None or pred_df.empty:
        return pd.DataFrame(columns=cols)
    if not globals().get("EVAL_ONLY_PWC_MATCHED_PAPERS", True):
        return pred_df
    pred_e = enrich_ours_combinations_with_dataset_json(
        pred_df, gt_json_path, match_threshold=match_threshold
    )
    if not len(gt_df) or "paper_match_key" not in gt_df.columns:
        return pred_df
    gt_keys = set(gt_df["paper_match_key"].dropna().unique())
    if not gt_keys:
        return pred_df
    out = pred_e[pred_e["paper_match_key"].isin(gt_keys)].copy()
    return out.drop(columns=["paper_match_key", "paper_match_kind"], errors="ignore")


def filter_pred_df_to_pwc_gt_vocab(pred_df: pd.DataFrame) -> pd.DataFrame:
    return pred_df



def _merge_tables_and_text_minimal(tables_df: pd.DataFrame, text_df: pd.DataFrame) -> pd.DataFrame:
    cols = ["paper_raw", "dataset_raw", "metric_raw", "paper_norm", "dataset_norm", "metric_norm"]
    parts = [df for df in (tables_df, text_df) if df is not None and not df.empty]
    if not parts:
        return pd.DataFrame(columns=cols)
    return pd.concat(parts, ignore_index=True).drop_duplicates()





def _bertscore_for_item_lists(
    pred_list: list[str],
    ref_list: list[str],
) -> tuple[float, float, float]:
    """P_bert, R_bert, F1_bert for one paper."""
    preds = [str(x).strip() for x in pred_list if str(x).strip()]
    refs = [str(x).strip() for x in ref_list if str(x).strip()]
    if not preds and not refs:
        return 1.0, 1.0, 1.0
    if not preds or not refs:
        return 0.0, 0.0, 0.0
    cands = [" ".join(preds)]
    references = [" ".join(refs)]
    P, R, F1 = bert_score_fn(
        cands, references, lang=_BERTSCORE_LANG, device=_BERTSCORE_DEVICE, verbose=False
    )
    return float(P[0]), float(R[0]), float(F1[0])


def _id_aware_bertscore_breakdown(
    pred_df: pd.DataFrame, gt_df: pd.DataFrame, gt_json_path: Path, thr: float,
    *, value_col: str = "metric_norm", agg_fn=None,
):
    """Id-aware matching: macro BERTScore + exact-match counts (n_intersection)."""
    if agg_fn is None:
        agg_fn = _agg_metric_list if value_col == "metric_norm" else _agg_dataset_list
    cols = ["paper_raw", "dataset_raw", "metric_raw", "paper_norm", "dataset_norm", "metric_norm"]
    pred_df = pred_df if pred_df is not None else pd.DataFrame(columns=cols)
    pred_e = enrich_ours_combinations_with_dataset_json(pred_df, gt_json_path, match_threshold=thr)

    ours_g = (
        pred_e.groupby("paper_match_key", sort=False)[value_col].apply(agg_fn)
        if len(pred_e) else pd.Series(dtype=object)
    )
    gt_g = (
        gt_df.groupby("paper_match_key", sort=False)[value_col].apply(agg_fn)
        if len(gt_df) else pd.Series(dtype=object)
    )

    gt_key_to_title = gt_df.groupby("paper_match_key", sort=False)["paper_raw"].first().to_dict() if len(gt_df) else {}
    pwc_key_list = list(gt_g.index)
    used_pwc: set[str] = set()
    key_map: dict[str, str] = {}

    for ok in ours_g.index:
        if ok in gt_g.index:
            key_map[ok] = ok
            used_pwc.add(ok)
            continue
        if not str(ok).startswith("title:"):
            key_map[ok] = ""
            continue
        best_pk, best_sc = "", 0.0
        for pk in pwc_key_list:
            if pk in used_pwc or not str(pk).startswith("title:"):
                continue
            sc = difflib.SequenceMatcher(None, str(ok), str(pk)).ratio()
            if sc > best_sc:
                best_sc, best_pk = sc, pk
        if best_pk and best_sc >= thr:
            key_map[ok] = best_pk
            used_pwc.add(best_pk)
        else:
            key_map[ok] = ""

    rep_norm = pred_e.groupby("paper_match_key", sort=False)["paper_norm"].first().to_dict() if len(pred_e) else {}
    all_keys = set(gt_g.index) | set(ours_g.index)
    detail_rows = []
    p_scores: list[float] = []
    r_scores: list[float] = []
    f1_scores: list[float] = []
    tp_total = 0

    pred_label = "pipeline_metrics" if value_col == "metric_norm" else "pipeline_datasets"
    gt_label = "ground_truth_metrics" if value_col == "metric_norm" else "ground_truth_datasets"

    for k in sorted(all_keys):
        ours_m = set(ours_g.get(k, [])) if k in ours_g.index else set()
        if k in ours_g.index:
            mk = key_map.get(k, "")
            gt_m = set(gt_g[mk]) if mk and mk in gt_g.index else set()
            paper_norm = rep_norm.get(k, str(k).replace("title:", "").strip())
            paper_title = gt_key_to_title.get(mk, "")
        else:
            gt_m = set(gt_g[k]) if k in gt_g.index else set()
            paper_norm = str(k).replace("title:", "").strip()
            paper_title = gt_key_to_title.get(k, "")

        if globals().get("EVAL_ONLY_PWC_MATCHED_PAPERS", True) and not gt_m:
            continue

        ours_list = sorted(ours_m)
        gt_list = sorted(gt_m)
        inter = ours_m & gt_m
        only_p = ours_m - gt_m
        only_g = gt_m - ours_m
        tp_total += len(inter)

        p_b, r_b, f1_b = (
            globals().get("_bertscore_for_item_lists")
            or globals().get("_bertscore_for_metric_lists")
        )(ours_list, gt_list)
        p_scores.append(p_b)
        r_scores.append(r_b)
        f1_scores.append(f1_b)

        detail_rows.append({
            "paper_norm": paper_norm,
            "paper_title": paper_title,
            pred_label: ", ".join(sorted(ours_m)),
            gt_label: ", ".join(sorted(gt_m)),
            "intersection": ", ".join(sorted(inter)),
            "only_pipeline": ", ".join(sorted(only_p)),
            "only_ground_truth": ", ".join(sorted(only_g)),
        })

    n_papers = len(f1_scores)
    summary = {
        "P_bert": round(sum(p_scores) / n_papers, 4) if n_papers else 0.0,
        "R_bert": round(sum(r_scores) / n_papers, 4) if n_papers else 0.0,
        "F1_bert": round(sum(f1_scores) / n_papers, 4) if n_papers else 0.0,
        "n_pipeline": int(sum(len(set(ours_g[k])) for k in ours_g.index)) if len(ours_g) else 0,
        "n_ground_truth": int(sum(len(set(gt_g[k])) for k in gt_g.index)) if len(gt_g) else 0,
        "n_intersection": int(tp_total),
    }
    return summary, pd.DataFrame(detail_rows)

print("PwC eval helpers ready (tables_only, text_only, tables_plus_text; BERTScore).")


In [ ]:
def run_ollama_extraction(extractor: OllamaKGEExtractor, model_slug: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, float, int]:
    combination_rows: list[dict] = []
    table_meta_rows: list[dict] = []
    chunk_calls = 0
    t0 = time.perf_counter()

    for pdf_stem, ocr_result in ocr_outputs.items():
        for page_data in ocr_result.get("results", []):
            page_num = int(page_data.get("page", 0) or 0)
            for table_idx, table in enumerate(page_data.get("tables", []), start=1):
                table_name = f"{pdf_stem}_p{page_num}_t{table_idx}"
                html = table.get("html", "")
                if not html:
                    continue
                header_lines, body_lines = _header_and_rows_from_html(html)
                if not header_lines and not body_lines:
                    continue

                table_sets: dict[str, set[str]] = {k: set() for k in LABELS}
                header_context = "\n".join(header_lines)

                def _run_chunk(prompt: str, kind: str, idx: int) -> None:
                    nonlocal chunk_calls
                    if DEBUG_LIMIT_CHUNKS is not None and chunk_calls >= DEBUG_LIMIT_CHUNKS:
                        return
                    text = prompt[:MAX_CHARS_PER_CHUNK]
                    cid = _chunk_id(table_name, kind, idx, text)
                    cache_p = _chunk_cache_path(model_slug, cid)
                    if USE_LLM_RESPONSE_CACHE and cache_p.is_file():
                        with open(cache_p, encoding="utf-8") as f:
                            ents = json.load(f)
                    else:
                        ents = extractor.extract_entities(text)
                        chunk_calls += 1
                        if ents is None:
                            ents = {"model": [], "dataset": [], "metric": []}
                        elif USE_LLM_RESPONSE_CACHE:
                            with open(cache_p, "w", encoding="utf-8") as f:
                                json.dump(ents, f, ensure_ascii=False, indent=0)
                    _merge_entity_sets(table_sets, ents)

                if header_context:
                    _run_chunk(f"Table column headers:\n{header_context}", "header", 0)
                for ri, row_text in enumerate(body_lines):
                    if header_context:
                        prompt = f"Table column headers: {header_context}\nRow: {row_text}"
                    else:
                        prompt = row_text
                    _run_chunk(prompt, "row", ri)
                if not any(table_sets[k] for k in LABELS):
                    full = "\n".join(header_lines + body_lines)
                    _run_chunk(full, "full", 0)

                _apply_gt_vocab_filter_to_entity_sets(pdf_stem, table_sets)
                combos = set(_combinations_from_sets(table_sets))
                table_meta_rows.append({
                    "paper": pdf_stem,
                    "table_name": table_name,
                    "models": " | ".join(sorted(table_sets["model"])) or "(none)",
                    "datasets": " | ".join(sorted(table_sets["dataset"])) or "(none)",
                    "metrics": " | ".join(sorted(table_sets["metric"])) or "(none)",
                    "combinations_found": len(combos),
                })
                for m, d, mt in sorted(combos):
                    combination_rows.append({
                        "paper": pdf_stem,
                        "table_name": table_name,
                        "model": m,
                        "dataset": d,
                        "metric": mt,
                    })

    combinations_df = pd.DataFrame(combination_rows)
    if combinations_df.empty:
        combinations_df = pd.DataFrame(columns=["paper", "table_name", "model", "dataset", "metric"])
    else:
        combinations_df = combinations_df.drop_duplicates().sort_values(
            ["paper", "table_name", "model", "dataset", "metric"]
        ).reset_index(drop=True)

    table_meta_df = pd.DataFrame(table_meta_rows)
    metric_table_rows = []
    twm = table_meta_df[table_meta_df["metrics"] != "(none)"] if len(table_meta_df) else table_meta_df
    for _, meta in twm.iterrows():
        paper, tname = meta["paper"], meta["table_name"]
        trips = combinations_df[
            (combinations_df["paper"] == paper) & (combinations_df["table_name"] == tname)
        ]
        if not trips.empty:
            for _, trip in trips.iterrows():
                metric_table_rows.append({
                    "paper": paper,
                    "table_name": tname,
                    "model": trip["model"],
                    "dataset": trip["dataset"],
                    "metric": trip["metric"],
                    "has_combination": True,
                })
        else:
            for mt in [x.strip() for x in str(meta["metrics"]).split("|") if x.strip()]:
                metric_table_rows.append({
                    "paper": paper,
                    "table_name": tname,
                    "model": "",
                    "dataset": "",
                    "metric": mt,
                    "has_combination": False,
                })
    tables_with_metrics_df = pd.DataFrame(metric_table_rows).drop_duplicates()

    elapsed = time.perf_counter() - t0
    return combinations_df, table_meta_df, tables_with_metrics_df, elapsed, chunk_calls



def _text_chunk_cache_path(model_slug: str, chunk_id: str) -> Path:
    d = LLM_CACHE_DIR / model_slug / "text"
    d.mkdir(parents=True, exist_ok=True)
    return d / f"{chunk_id}.json"


def _arxiv_norm_from_text(text: object) -> str:
    """Same id as eval Section 6; used for Section 5 TEI fallback when stem≠filename."""
    if text is None:
        return ""
    s = str(text).strip().lower()
    m = re.search(r"(\d{4}\.\d{4,5})(?:v\d+)?", s)
    return m.group(1) if m else ""


def _resolve_xml_for_pdf_stem(
    repo_root: Path,
    pdf_stem: str,
    *,
    xml_rel_dir: Path | None = None,
) -> Path | None:
    """Resolve GROBID TEI XML for a PDF stem (xml_files_3: <stem>.tei.xml)."""
    stem = str(pdf_stem).strip()
    if not stem:
        return None
    xml_dir = (repo_root / (xml_rel_dir or Path("data/xml_files_3"))).resolve()
    if not xml_dir.is_dir():
        return None
    direct = xml_dir / f"{stem}.tei.xml"
    if direct.is_file():
        return direct
    a = _arxiv_norm_from_text(stem)
    if a:
        hits = sorted(xml_dir.glob(f"*{a}*.xml"))
        if hits:
            return hits[0].resolve()
    return None


def _tei_narrative_chunks_from_path(
    xml_path: Path,
    section_keywords: list[str],
    scan_full_body: bool,
    max_chars: int | None = None,
) -> list[str]:
    """Fragmentos de texto narrativo TEI (sin celdas <table>) para GLiNER/Ollama."""
    from bs4 import BeautifulSoup

    limit = max_chars if max_chars is not None else MAX_CHARS_PER_CHUNK
    with open(xml_path, encoding="utf-8", errors="replace") as f:
        soup = BeautifulSoup(f, "lxml-xml")
    body = soup.find("body")
    if body is None:
        return []
    keywords = [k.lower() for k in section_keywords if k]
    chunks: list[str] = []
    for div in body.find_all("div"):
        head = div.find("head")
        if not head:
            continue
        title = head.get_text(strip=True).lower()
        if keywords and not any(kw in title for kw in keywords):
            continue
        txt = div.get_text(separator=" ", strip=True)
        if txt:
            chunks.append(txt[:limit])
    if scan_full_body and not chunks:
        txt = body.get_text(separator=" ", strip=True)
        if txt:
            chunks.append(txt[:limit])
    return chunks



def _apply_gt_vocab_filter_to_entity_sets(pdf_stem: str, entity_sets: dict[str, set[str]]) -> None:
    return  # minimal: no GT vocab filter


def _metrics_rows_from_entity_sets(pdf_stem: str, table_name: str, entity_sets: dict[str, set[str]]) -> list[dict]:
    """Rows for Tables/Text With Metrics sheets used in evaluation."""
    rows: list[dict] = []
    combos = set(_combinations_from_sets(entity_sets))
    if combos:
        for m, d, mt in sorted(combos):
            rows.append({
                "paper": pdf_stem,
                "table_name": table_name,
                "model": m,
                "dataset": d,
                "metric": mt,
                "has_combination": True,
            })
    else:
        for mt in sorted(entity_sets.get("metric", set())):
            rows.append({
                "paper": pdf_stem,
                "table_name": table_name,
                "model": "",
                "dataset": "",
                "metric": mt,
                "has_combination": False,
            })
    return rows


def run_ollama_text_extraction(
    extractor: OllamaKGEExtractor, model_slug: str
) -> tuple[pd.DataFrame, float, int]:
    """text_only: Ollama sobre narrativa TEI (xml_files_3), misma lógica de entidades que tablas."""
    t0 = time.perf_counter()
    chunk_calls = 0
    all_rows: list[dict] = []

    stems = list(ocr_outputs.keys())
    if MAX_PAPERS:
        stems = stems[:MAX_PAPERS]

    for pdf_stem in stems:
        xml_path = _resolve_xml_for_pdf_stem(REPO, pdf_stem, xml_rel_dir=XML_FILES_DIR)
        if xml_path is None:
            continue
        chunks = _tei_narrative_chunks_from_path(
            xml_path,
            GT_XML_SECTION_KEYWORDS,
            GT_SCAN_FULL_BODY_IF_EMPTY,
            MAX_CHARS_PER_CHUNK,
        )
        if not chunks:
            continue
        paper_sets: dict[str, set[str]] = {k: set() for k in LABELS}
        table_name = f"{pdf_stem}_text"

        for si, chunk_text in enumerate(chunks):
            if DEBUG_LIMIT_CHUNKS is not None and chunk_calls >= DEBUG_LIMIT_CHUNKS:
                break
            cid = _chunk_id(table_name, "tei", si, chunk_text)
            cache_p = _text_chunk_cache_path(model_slug, cid)
            if USE_LLM_RESPONSE_CACHE and cache_p.is_file():
                with open(cache_p, encoding="utf-8") as f:
                    ents = json.load(f)
            else:
                ents = extractor.extract_entities(
                    f"Article text (experiments/results section):\n{chunk_text}"
                )
                chunk_calls += 1
                if ents is None:
                    ents = {"model": [], "dataset": [], "metric": []}
                elif USE_LLM_RESPONSE_CACHE:
                    with open(cache_p, "w", encoding="utf-8") as f:
                        json.dump(ents, f, ensure_ascii=False, indent=0)
            _merge_entity_sets(paper_sets, ents)

        _apply_gt_vocab_filter_to_entity_sets(pdf_stem, paper_sets)
        all_rows.extend(_metrics_rows_from_entity_sets(pdf_stem, table_name, paper_sets))

    text_df = pd.DataFrame(all_rows).drop_duplicates() if all_rows else pd.DataFrame(
        columns=["paper", "table_name", "model", "dataset", "metric", "has_combination"]
    )
    return text_df, time.perf_counter() - t0, chunk_calls


def load_pred_text_only_sheet(excel_path: Path) -> pd.DataFrame:
    """Predicciones text_only de la hoja Text With Metrics (misma forma que tablas)."""
    cols = [
        "paper_raw", "dataset_raw", "metric_raw",
        "paper_norm", "dataset_norm", "metric_norm",
    ]
    try:
        df = pd.read_excel(excel_path, sheet_name="Text With Metrics")
    except ValueError:
        return pd.DataFrame(columns=cols)
    colmap = {c.lower(): c for c in df.columns}
    if "paper" not in colmap or "metric" not in colmap:
        return pd.DataFrame(columns=cols)
    part = pd.DataFrame({
        "paper_raw": df[colmap["paper"]],
        "dataset_raw": df[colmap["dataset"]] if "dataset" in colmap else "",
        "metric_raw": df[colmap["metric"]],
    })
    part["paper_norm"] = part["paper_raw"].map(normalize_paper)
    part["dataset_norm"] = part["dataset_raw"].map(normalize_dataset)
    part["metric_norm"] = part["metric_raw"].map(normalize_metric)
    part = part[(part["paper_norm"] != "") & (part["metric_norm"] != "")]
    return part.drop_duplicates()


print("Text extraction helpers (Ollama on TEI) ready.")

print("run_ollama_extraction() + text extraction defined.")


In [ ]:
OLLAMA_EXTRACTION_RUNS: list[dict] = []

for ollama_tag, model_slug in OLLAMA_MODELS:
    print("=" * 70)
    print(f"OLLAMA MODEL: {ollama_tag}")
    print("=" * 70)
    extractor = OllamaKGEExtractor(ollama_tag)
    extractor.warmup()

    comb_df, meta_df, twm_df, llm_seconds, n_chunks = run_ollama_extraction(extractor, model_slug)
    text_df, text_seconds, text_chunks = run_ollama_text_extraction(extractor, model_slug)

    out_xlsx = TABLE_EXTRACTION_DIR / f"ollama_{model_slug}_lightonocr_combinations_{CORPUS_TAG}.xlsx"
    comb_x = _sanitize_df_for_excel(comb_df)
    meta_x = _sanitize_df_for_excel(meta_df)
    twm_x = _sanitize_df_for_excel(twm_df)
    text_x = _sanitize_df_for_excel(text_df)
    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
        comb_x.to_excel(writer, sheet_name="Combinations", index=False)
        meta_x.to_excel(writer, sheet_name="Table Metadata", index=False)
        twm_x.to_excel(writer, sheet_name="Tables With Metrics", index=False)
        text_x.to_excel(writer, sheet_name="Text With Metrics", index=False)

    row = {
        "ollama_model": ollama_tag,
        "model_slug": model_slug,
        "output_excel": str(out_xlsx.relative_to(REPO)),
        "llm_seconds": round(llm_seconds, 2),
        "llm_chunk_calls": n_chunks,
        "llm_text_seconds": round(text_seconds, 2),
        "llm_text_chunk_calls": text_chunks,
        "n_combinations": len(comb_df),
        "n_tables_with_metrics_rows": len(twm_df),
        "n_text_with_metrics_rows": len(text_df),
        "n_papers": comb_df["paper"].nunique() if len(comb_df) else 0,
    }
    OLLAMA_EXTRACTION_RUNS.append(row)
    print(f"Tables: combinations={len(comb_df)} | Tables With Metrics={len(twm_df)} | {llm_seconds:.1f}s ({n_chunks} calls)")
    print(f"Text:   Text With Metrics={len(text_df)} | {text_seconds:.1f}s ({text_chunks} calls)")
    print(f"Written: {out_xlsx}")
    display(comb_df.head(8))

print(f"\nExtraction done ({len(OLLAMA_EXTRACTION_RUNS)} models). Run Section 6b for evaluation.")


## 6b — Evaluation (three modes per Ollama model)

`tables_only` reads sheet Tables With Metrics. `text_only` reads Text With Metrics. `tables_plus_text` merges both.

Re-run Section 5 if combination Excel files lack the text sheet.

Writes `ollama_metric_eval_allpapers.xlsx` and `ollama_metric_audit_allpapers.xlsx`. See `README_pwc_benchmarks.md`.


In [ ]:
# Full-corpus audit (all papers), aligned with GLiNER; three modes per Ollama model
import time as _time_eval

try:
    _pdf_rel = PDF_FILES_DIR.relative_to(REPO)
except ValueError:
    _pdf_rel = Path("data") / PDF_FILES_DIR.name if isinstance(PDF_FILES_DIR, Path) else Path("data/pdf_files_3")
_gt_json_path = (REPO / "data" / PWC_GT_JSON).resolve()
_match_thr = float(MATCH_THRESHOLD)

print("[eval] Building GT from pwc_final.json …")
_gt_df_all, _, _ = build_ground_truth_for_pdf_corpus(
    REPO, PWC_GT_JSON, _pdf_rel, match_threshold=_match_thr, skip_empty_metrics=True,
)

_runs = list(globals().get("OLLAMA_EXTRACTION_RUNS") or [])
if not _runs:
    for ollama_tag, model_slug in OLLAMA_MODELS:
        pred_xlsx = TABLE_EXTRACTION_DIR / f"ollama_{model_slug}_lightonocr_combinations_{CORPUS_TAG}.xlsx"
        if pred_xlsx.is_file():
            _runs.append({"ollama_model": ollama_tag, "model_slug": model_slug})

if not _runs:
    raise FileNotFoundError("No Ollama extraction found; run Section 5 before this cell.")

summary_rows: list[dict] = []
audit_parts: list[pd.DataFrame] = []

for _run in _runs:
    ollama_tag = str(_run.get("ollama_model", ""))
    model_slug = str(_run.get("model_slug", ""))
    pred_xlsx = TABLE_EXTRACTION_DIR / f"ollama_{model_slug}_lightonocr_combinations_{CORPUS_TAG}.xlsx"
    if not pred_xlsx.is_file():
        print(f"SKIP (missing): {pred_xlsx}")
        continue

    _sec_tables = float(_run.get("llm_seconds", 0) or 0)
    _sec_text = float(_run.get("llm_text_seconds", 0) or 0)
    _cache_only = (
        int(_run.get("llm_chunk_calls", 0) or 0) == 0
        and int(_run.get("llm_text_chunk_calls", 0) or 0) == 0
    )

    _pred_tables = load_pred_tables_only_sheet(pred_xlsx)
    _pred_text = load_pred_text_only_sheet(pred_xlsx)
    _pred_tables = _filter_pred_df_to_gt_corpus(_pred_tables, _gt_df_all, _gt_json_path, _match_thr)
    _pred_text = _filter_pred_df_to_gt_corpus(_pred_text, _gt_df_all, _gt_json_path, _match_thr)
    _pred_tables = _df_to_pred_metric(_pred_tables)
    _pred_text = _df_to_pred_metric(_pred_text)
    _pred_union = _merge_tables_and_text_minimal(_pred_tables, _pred_text)

    _reported = globals().get("OLLAMA_REPORTED_INFERENCE_SECONDS") or {}
    mode_inputs = [
        ("tables_only", _pred_tables, _sec_tables),
        ("text_only", _pred_text, _sec_text),
        ("tables_plus_text", _pred_union, _sec_tables + _sec_text),
    ]

    for mode_name, pred_df, sec in mode_inputs:
        if _cache_only:
            sec = float(_reported.get((ollama_tag, mode_name), sec))
        if mode_name == "text_only" and not EVAL_SETUP_TEXT_ONLY:
            continue
        if mode_name == "tables_only" and not EVAL_SETUP_TABLES_ONLY:
            continue
        if mode_name == "tables_plus_text" and not EVAL_SETUP_TABLES_PLUS_TEXT:
            continue
        summ, det = _id_aware_bertscore_breakdown(pred_df, _gt_df_all, _gt_json_path, _match_thr, value_col="metric_norm", agg_fn=_agg_metric_list)
        summary_rows.append({
            "ollama_model": ollama_tag,
            "mode": mode_name,
            "P_bert": summ["P_bert"],
            "R_bert": summ["R_bert"],
            "F1_bert": summ["F1_bert"],
            "time_seconds": round(sec, 2),
            "n_pipeline": summ["n_pipeline"],
            "n_ground_truth": summ["n_ground_truth"],
            "n_intersection": summ["n_intersection"],
        })
        det = det.copy()
        det.insert(0, "ollama_model", ollama_tag)
        det.insert(1, "mode", mode_name)
        audit_parts.append(det)
        print(
            f"[{ollama_tag} | {mode_name}] P_bert={summ['P_bert']:.4f} "
            f"R_bert={summ['R_bert']:.4f} F1_bert={summ['F1_bert']:.4f}"
        )

eval_df = pd.DataFrame(summary_rows)
audit_df = pd.concat(audit_parts, ignore_index=True) if audit_parts else pd.DataFrame()

if len(audit_df):
    audit_df = audit_df[[
        "ollama_model", "paper_norm", "paper_title", "mode",
        "pipeline_metrics", "ground_truth_metrics", "intersection",
        "only_pipeline", "only_ground_truth",
    ]].copy()
    audit_df = audit_df.rename(columns={"paper_norm": "paper", "paper_title": "title"})
    _mode_order = {"tables_only": 0, "text_only": 1, "tables_plus_text": 2}
    audit_df["_paper_k"] = audit_df["paper"].astype(str)
    audit_df["_mode_k"] = audit_df["mode"].map(_mode_order).fillna(9)
    audit_df = audit_df.sort_values(["ollama_model", "_paper_k", "_mode_k"], kind="stable").drop(
        columns=["_paper_k", "_mode_k"]
    ).reset_index(drop=True)

with pd.ExcelWriter(OLLAMA_EVAL_ALL_XLSX, engine="openpyxl") as _w:
    eval_df.to_excel(_w, sheet_name="evaluation", index=False)
with pd.ExcelWriter(OLLAMA_AUDIT_ALL_XLSX, engine="openpyxl") as _w:
    audit_df.to_excel(_w, sheet_name="audit_all_papers", index=False)

_use_gt_vocab = globals().get("FILTER_PREDICTIONS_TO_PWC_GT_VOCAB", False)
_eval_matched = globals().get("EVAL_ONLY_PWC_MATCHED_PAPERS", True)
print(f"Minimal pipeline: FILTER_PREDICTIONS_TO_PWC_GT_VOCAB={_use_gt_vocab}, BERTScore eval")
print(f"EVAL_ONLY_PWC_MATCHED_PAPERS={_eval_matched}")
print(f"Evaluation Excel: {OLLAMA_EVAL_ALL_XLSX.resolve()}")
print(f"Audit Excel: {OLLAMA_AUDIT_ALL_XLSX.resolve()}")
display(eval_df)
display(audit_df.head(30))
